# 01 — EDA (M4)
Run `02_cleaning` (or the pipeline) first so `data/processed/flowcast_clean.parquet` exists.

In [ ]:
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np
import pandas as pd
from src.utils import load_config, set_seed
config = load_config()
set_seed(config["seed"])
RAW_DIR = PROJECT_ROOT / "data" / "raw"

In [ ]:
clean = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'flowcast_clean.parquet')
print(clean.shape)
clean.head()

## Descriptive statistics (mean, median, std) — observed values only

In [ ]:
from src.eda import summary_statistics
summary_statistics(clean).round(2)

## Daily profile
Expected: two weekday peaks (morning and evening), a flatter weekend curve.

In [ ]:
from src.eda import daily_profile_figure
profile = daily_profile_figure(clean, config['paths']['figures_dir'])
from IPython.display import Image
Image(str(config['paths']['figures_dir'] / 'eda_daily_profile.png'))

## Class balance — why macro-F1 and class weights

In [ ]:
from src.eda import congestion_balance_figure
congestion_balance_figure(clean, config, config['paths']['figures_dir'])

## Accident rate — why ROC-AUC + PR-AUC, never accuracy
Expected positive rate ≈ 0.9 %.

In [ ]:
known = clean.loc[clean['accident_observed']]
print(f"positive rate: {(known['accident_count'] > 0).mean():.3%}")

## Weather vs traffic

In [ ]:
from src.eda import weather_effect_table
weather_effect_table(clean).round(2)

## Correlation with next-window volume (train rows)
r = cov(x, y) / (σx σy). Expected: the latest volume (vol_t0) and same-window-last-week lags lead.

In [ ]:
from src.features import build_features
from src.eda import correlation_figure
features_h1 = build_features(clean, config, 1)
correlation_figure(features_h1, config['paths']['figures_dir']).head(10)